In [ ]:
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from dotenv import load_dotenv

In [ ]:
import vertexai

from vertexai.generative_models import GenerativeModel, GenerationConfig
from google.oauth2 import service_account

In [ ]:
load_dotenv()

In [ ]:
GOOGLE_CLOUD_SERVICE_ACCOUNT_PATH=os.getenv("GOOGLE_CLOUD_SERVICE_ACCOUNT_PATH")
GOOGLE_CLOUD_REGION=os.getenv("GOOGLE_CLOUD_REGION")
GOOGLE_CLOUD_PROJECT_ID=os.getenv("GOOGLE_CLOUD_PROJECT_ID")
print(f"{GOOGLE_CLOUD_SERVICE_ACCOUNT_PATH}\n{GOOGLE_CLOUD_REGION}\n{GOOGLE_CLOUD_PROJECT_ID}")

In [ ]:
credentials = service_account.Credentials.from_service_account_file(
    GOOGLE_CLOUD_SERVICE_ACCOUNT_PATH
)

In [ ]:
vertexai.init(project="silrokai", location="asia-northeast3", credentials=credentials)

In [ ]:
model = GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=GenerationConfig(
        temperature=0.2,
        max_output_tokens=2048,
    )
)

In [ ]:
def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
import re

def extract_tagged_output(text: str):
    context_match = re.search(r"<context>(.*?)</context>", text, re.DOTALL)
    agenda_match = re.search(r"<agenda>(.*?)</agenda>", text, re.DOTALL)

    context = context_match.group(1).strip() if context_match else None
    agenda_raw = agenda_match.group(1).strip() if agenda_match else ""
    agenda = [int(a) for a in agenda_raw.split(",")] if agenda_raw else []

    return context, agenda

In [ ]:
def print_tagged_output(cnt_msg:str, response: str, agenda:list[str]):
    print("현재 메시지" + "-" * 40)
    print(cnt_msg)
    print("응답" + "-" * 10)
    print(response)

    context, agenda_ = extract_tagged_output(response)
    if context:
        print("Context:" + "-" * 10)
        print(context)
    else:
        print("No context found.")

    if agenda_:
        print("\nAgenda:" + "-" * 10)
        print(agenda_)
        agenda = [a for i, a in enumerate(agenda) if i not in agenda_]
    else:
        print("No agenda found.")
    return agenda

In [ ]:
def run_chat(conversation:list[str], prompt:str, last_prompt:str, agenda:list[str]):
    chat = model.start_chat(history=[], response_validation=False)

    cnt_msg = ""
    for message in conversation:
        cnt_msg += message + "\n"
        if len(cnt_msg) > 500:
            agenda_text = ""
            for i, a in enumerate(agenda):
                agenda_text += f"{i}: {a}\n"
            cnt_msg = prompt.format(agenda_text, cnt_msg)
            response = chat.send_message(cnt_msg)
            agenda = print_tagged_output(cnt_msg, response.text, agenda)
            cnt_msg = ""

    if cnt_msg:
        agenda_text = ""
        for i, a in enumerate(agenda):
            agenda_text += f"{i}: {a}\n"
        cnt_msg = prompt.format(agenda_text, cnt_msg)
        response = chat.send_message(cnt_msg)
        print_tagged_output(cnt_msg, response.text, agenda)

    print(f"최종 프롬프트 전송" + "-" * 40)
    response = chat.send_message(last_prompt)
    print(response.text)
    print("-"*40)

In [ ]:
prompt = (
"""
너는 회의 내용을 정리해주는 뉴스 아나운서야.

[추론 지침]
- 최대한 빠르게 출력해줘

[요약 지침]
- 이번 회의 내용 일부만 읽고, 핵심만 한 문장으로 요약해.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 작성해.
- 마크다운, 리스트, 기호 없이 일반 텍스트로 작성해.
- 반드시 다음 형식으로 출력해:
  <context>요약 내용</context>

[아젠다 판별 지침]
- 아래 아젠다 항목 중, 이번 회의 일부 내용에 의해 **명확히 완료된 항목**이 있다면 번호를 출력해.
- “완료”란, 실제로 실행되었거나 결정되었다는 표현이 명확히 포함된 경우만 의미함.
- 단순히 언급만 되었거나 논의 중인 것은 완료로 간주하지 마.
- 관련된 아젠다가 없다면 빈 태그로 남겨둬.
- 반드시 다음 형식으로 출력해:
  <agenda>1, 3</agenda>

[출력 지침]
- 먼저 요약 내용을 <context> 태그로 출력해.
- 다음으로 완료된 아젠다 항목 번호를 <agenda> 태그로 출력해.
- 최대 토큰이 2048이야. 이내로 출력해줘.

[아젠다 항목]
{0}

[회의 내용 일부]
{1}
"""
)

last_prompt = (
"""
너는 전체 회의 내용을 정리해주는 뉴스 아나운서야.

[요약 지침]
- 지금까지 나눈 회의 내용을 읽고, 핵심만 뽑아 요약해.
- 전체 내용을 100문장 이내로 정리하되, 주제 흐름에 따라 자연스럽게 구성해.
- 말투는 뉴스 보도처럼 객관적이고 간결하게 작성해.
- 마크다운, 리스트, 기호 없이 일반 텍스트로 작성해.
- 반드시 다음 형식으로 출력해:
  <context>요약 내용</context>
"""
)


In [ ]:
text = None
with open("/workspaces/dev/.storage/meeting_sample.txt", "r") as f:
    text = f.readlines()

In [ ]:
agenda = [
    "회의 목적 설명",
    "프로젝트 진행 상황 공유",
    "이슈 및 장애물 논의",
    "다음 회의 일정 조율"
]


In [ ]:
run_chat(
    conversation=text,
    prompt=prompt,
    last_prompt=last_prompt,
    agenda=agenda
)